# Обучение моделей

В данном ноутбуке выполняется обучение ML-моделей на подготовленных данных.

Данные были получены после этапа Feature Engineering:

На данном этапе:
- загружаются готовые `X_train`, `y_train`, `X_test`, `y_test`;
- обучаются различные модели;
- оценивается качество по ROC-AUC;
- лучшие модели сохраняются для дальнейшего сравнения и оценки качества рекомендательной системы по метрике HitRate@5.

In [1]:
import pandas as pd
import pickle

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")["target"]

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")["target"]

In [2]:
#Переменная для сохранения результов моделей 
results = []

## Logistic Regression baseline

В качестве 1 модели используется Logistic Regression.

Модель выбрана как baseline для оценки качества сформированных признаков.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import gc


model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        solver='saga',
        max_iter=1000,
        verbose=1,
        n_jobs=-1,
        random_state=42
    ))
])

model.fit(X_train, y_train)


gc.collect()
y_train_proba = model.predict_proba(X_train)[:, 1]
y_test_proba = model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

results.append({
    "model": "LogisticRegression",
    "train_auc": train_auc,
    "test_auc": test_auc
})


with open("../models/logistic_regression.pkl", "wb") as file:
    pickle.dump(model, file)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


convergence after 37 epochs took 147 seconds
Train AUC: 0.6607
Test AUC: 0.6164


## CatBoost

В качестве 2 модели используется CatBoostClassifier.

CatBoost выбран благодаря способности:
- работать с нелинейными зависимостями между признаками;

In [4]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function='Logloss',
    verbose=100
)

model.fit(
    X_train,
    y_train
)


gc.collect()
y_train_proba = model.predict_proba(X_train)[:, 1]
y_test_proba = model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

results.append({
    "model": "CatBoostClassifier",
    "train_auc": train_auc,
    "test_auc": test_auc
})

with open("../models/catboost.pkl", "wb") as file:
    pickle.dump(model, file)

0:	learn: 0.6440576	total: 717ms	remaining: 5m 58s
100:	learn: 0.2976532	total: 59.3s	remaining: 3m 54s
200:	learn: 0.2958801	total: 1m 55s	remaining: 2m 51s
300:	learn: 0.2951171	total: 2m 50s	remaining: 1m 52s
400:	learn: 0.2945089	total: 3m 43s	remaining: 55.2s
499:	learn: 0.2940597	total: 4m 37s	remaining: 0us
Train AUC: 0.7245
Test AUC: 0.6603


## LightGBM

В качестве 3 модели используется LightGBM.

LightGBM выбран как один из наиболее эффективных алгоритмов для табличных данных.

Преимущества модели:
- высокая скорость обучения;
- хорошая работа с большим количеством признаков;

In [5]:
from lightgbm import LGBMClassifier


model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train,
    y_train
)


gc.collect()


y_train_proba = model.predict_proba(X_train)[:, 1]
y_test_proba = model.predict_proba(X_test)[:, 1]


train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)


print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

results.append({
    "model": "LGBMClassifier",
    "train_auc": train_auc,
    "test_auc": test_auc
})


with open("../models/lightgbm.pkl", "wb") as file:
    pickle.dump(model, file)

Train AUC: 0.7238
Test AUC: 0.6590


## Результаты сравнения моделей

После обучения моделей было выполнено сравнение качества по метрике ROC-AUC.

В эксперименте участвовали:
- Logistic Regression — baseline-модель;
- CatBoostClassifier;
- LightGBM;
- RecMLP Neural Network (обучение выполнялось в отдельном ноутбуке `03_neural_network.ipynb`).

По результатам сравнения лучшую метрику на тестовой выборке показала модель **CatBoostClassifier** с ROC-AUC = 0.6603.

LightGBM показал сопоставимое качество (ROC-AUC = 0.6590), а нейронная сеть RecMLP достигла ROC-AUC = 0.6576.

Несмотря на то, что RecMLP Neural Network показала меньшее качество по сравнению с CatBoost, модель имеет потенциал для дальнейшего улучшения.

Возможные направления улучшения:
- подбор архитектуры нейронной сети;
- настройка количества слоёв и размеров скрытых представлений;
- подбор гиперпараметров обучения (learning rate, batch size, количество эпох);
- использование более сложных методов регуляризации;
- применение embedding-представлений для категориальных признаков.

В качестве основной модели для рекомендательной системы выбирается **CatBoostClassifier**.

Дополнительно было выполнено сравнение моделей по метрике рекомендательной системы **HitRate@5**. Результаты сравнения находятся в отдельном ноутбуке. `04_hitrate_comparison.ipynb`

In [6]:
#добавленны результаты Neural Network
results.append({
    "model": "RecMLP Neural Network",
    "train_auc": 0.7160,
    "test_auc": 0.6576
})

results_df = pd.DataFrame(results)

# Сортировка по качеству на тесте
results_df = results_df.sort_values(
    by="test_auc",
    ascending=False
).reset_index(drop=True)

# Добавляем место в рейтинге
results_df.index = results_df.index + 1
results_df.index.name = "Rank"

results_df

,model,train_auc,test_auc
Rank,,,
1,CatBoostClassifier,0.724479,0.660270
2,LGBMClassifier,0.723791,0.658969
3,RecMLP Neural Network,0.716000,0.657600
4,LogisticRegression,0.660694,0.616415
